# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL and defines multiple record sets, fields, and columns.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show basic info
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and columns using the `@id` identifiers.

We print all record sets and fields defined in the metadata, showing their `@id` and `name`.

In [ ]:
# Get all record sets with their @id and display their fields
record_sets = dataset.metadata.record_sets

print(f"Number of record sets: {len(record_sets)}\n")

record_set_ids = []
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    Field @id: {f.id} | name: {f.name} | dataType: {f.data_type}")
    print("")
    record_set_ids.append(rs.id)
# For demonstration, choose the first record set for further work
if len(record_set_ids) > 0:
    example_record_set_id = record_set_ids[0]
else:
    raise ValueError("No record sets found in the metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. The `@id` fields identified above are used.

*This cell demonstrates loading each record set, printing its columns, and showing the first rows for a selected record set.*

In [ ]:
# Extract data from all record sets
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(f"  Number of records: {len(df)}\n")
        else:
            print("  No records found for this record set.\n")
    except Exception as e:
        print(f"  Error loading records: {e}\n")
    print()
print(f"Example DataFrame for record set '{example_record_set_id}':\n")
example_df = dataframes[example_record_set_id]
print(example_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by a numeric field, normalization, and optionally grouping by a categorical field.

*Modify the `numeric_field_id` and `group_field_id` below based on the overview above (using the field `@id`).*

In [ ]:
# Choose a numeric field, e.g., molecular weight, peptide_count, or coverage_percentage by field @id
# Use the first numeric field found from the chosen record set
selected_record_set_id = example_record_set_id
fields = [f for f in dataset.metadata.find_record_set(selected_record_set_id).fields]
numeric_field_id = None
numeric_types = {"Integer", "Float", "Number"}
for f in fields:
    if (f.data_type in numeric_types) and (f.id in example_df.columns):
        numeric_field_id = f.id
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found in the example record set.")

# Filtering and normalization
threshold = example_df[numeric_field_id].mean()  # Use mean as an example threshold
filtered_df = example_df[example_df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by first categorical field if available
group_field_id = None
for f in fields:
    if f.data_type == "Text" and f.id in example_df.columns:
        group_field_id = f.id
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("\nNo suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version using basic histograms.

Further visualizations could include pairwise relationships (scatter plots) or group-comparison bar plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True, color='skyblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (filtered)")
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True, color='salmon')
plt.xlabel(f"{numeric_field_id}_normalized")
plt.title(f"Normalized {numeric_field_id} (filtered)")
plt.show()

# Optionally visualize group means if group_field_id exists
if group_field_id:
    plt.figure(figsize=(10, 4))
    grouped_df.plot(kind='bar')
    plt.xlabel(group_field_id)
    plt.ylabel(f"mean({numeric_field_id})")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded Croissant metadata and records using `mlcroissant`, referencing all entities by their `@id`.
- Explored record sets, fields, and columns.
- Loaded dataframes for each record set and selected meaningful fields for analysis.
- Filtered and normalized numeric data, and performed simple groupby aggregation if appropriate.
- Visualized the results via histograms and bar plots.

Further analysis can use the full richness of the dataset—refer to the field and record `@id`s for precise, reproducible data extraction and processing.